# Poglavje 7: Izdelava klepetalnih aplikacij
## Hiter začetek z API-jem Microsoft Foundry Models

Ta zvezek je prirejen iz [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ki vključuje zvezke za dostop do storitev [Azure OpenAI](aoai-assignment.ipynb).

> **Opomba:** GitHub Models se upokojuje konec julija 2026. Ta zvezek zdaj cilja na [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), ki ponuja enak brezplačen katalog modelov za preizkušanje in izkušnjo Azure AI Inference SDK.


# Pregled  
"Veliki jezikovni modeli so funkcije, ki preslikajo besedilo v besedilo. Glede na vhodni niz besedila velik jezikovni model poskuša napovedati besedilo, ki bo sledilo"(1). Ta "quickstart" zvezek bo uporabnike seznanil s koncepti visokega nivoja LLM, osnovnimi zahtevami paketov za začetek z AML, nežnim uvodom v oblikovanje pozivov in nekaj kratkimi primeri različnih primerov uporabe. 


## Kazalo vsebine  

[Pregled](#overview)  
[Kako uporabljati OpenAI storitev](#how-to-use-openai-service)  
[1. Ustvarjanje vaše OpenAI storitve](#1.-creating-your-openai-service)  
[2. Namestitev](#2.-installation)    
[3. Prijavni podatki](#3.-credentials)  

[Uporabni primeri](#use-cases)    
[1. Povzetek besedila](#1.-summarize-text)  
[2. Razvrščanje besedila](#2.-classify-text)  
[3. Ustvarjanje novih imen izdelkov](#3.-generate-new-product-names)  
[4. Natančna nastavitev klasifikatorja](#4.fine-tune-a-classifier)  

[Reference](#references)


### Ustvarite svoj prvi poziv  
Ta kratek vodnik bo zagotovil osnovni uvod za pošiljanje pozivov modelu v Microsoft Foundry Models za preprosto nalogo "povzemanje".  


**Koraki**:  
1. Namestite knjižnico `azure-ai-inference` v vaše Python okolje, če tega še niste storili.  
2. Naložite standardne pomožne knjižnice in nastavite svoje poverilnice za Microsoft Foundry Models.  
3. Izberite model za svojo nalogo  
4. Ustvarite preprost poziv za model  
5. Pošljite svoj zahtevek prek API-ja modela!


### 1. Namesti `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Uvozite pomožne knjižnice in ustvarite poverilnice


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Iskanje pravega modela  
Modeli, kot so GPT-4o in GPT-4o mini, lahko razumejo in ustvarjajo naravni jezik ter so na voljo v katalogu Microsoft Foundry Models skupaj z modeli iz Meta, Mistral, Cohere in Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Oblikovanje pozivov  

"Čar velikih jezikovnih modelov je v tem, da se s treniranjem za minimizacijo te napake napovedovanja čez ogromne količine besedila modeli na koncu naučijo konceptov, uporabnih za te napovedi. Na primer, naučijo se konceptov, kot so"(1):

* kako črkovati
* kako deluje slovnica
* kako parafrazirati
* kako odgovarjati na vprašanja
* kako voditi pogovor
* kako pisati v mnogih jezikih
* kako programirati
* itd.

#### Kako nadzorovati velik jezikovni model  
"Od vseh vhodov v velik jezikovni model je daleč najbolj vpliven prav besedilni poziv(1).

Velike jezikovne modele lahko spodbudimo k ustvarjanju izhoda na nekaj načinov:

Navodilo: Povej modelu, kaj želiš
Dokončanje: Zbudi model, da dokonča začetek tistega, kar želiš
Demonstracija: Pokaži modelu, kaj želiš, z enim od teh načinov:
Nekaj primerov v pozivu
Mnoge stotine ali tisoči primerov v naboru podatkov za fino nastavljanje"



#### Obstajajo tri osnovna pravila za ustvarjanje pozivov:

**Pokaži in povej**. Jasno povej, kaj želiš, bodisi skozi navodila, primere ali kombinacijo obeh. Če želiš, da model razvrsti seznam predmetov po abecednem vrstnem redu ali da razvrsti odstavek glede na čustvenost, mu pokaži, da je to tisto, kar želiš.

**Nudi kakovostne podatke**. Če poskušaš izdelati klasifikator ali doseči, da model sledi vzorcu, poskrbi, da je dovolj primerov. Preveri svoje primere — model je ponavadi dovolj pameten, da prepozna osnovne pravopisne napake in ti da odgovor, a lahko tudi predpostavi, da je to namerno, kar lahko vpliva na odgovor.

**Preveri nastavitve.** Nastavitve temperature in top_p nadzorujejo, kako determinističen je model pri generiranju odgovora. Če zahtevaš odgovor, kjer obstaja samo en pravilen odgovor, jih nastavi na nižje vrednosti. Če želiš bolj raznolike odgovore, jih nastavi višje. Največja napaka, ki jo ljudje delajo pri teh nastavitvah, je, da mislijo, da gre za kontrola "pameti" ali "kreativnosti".


Vir: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Oddajte!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ponovite isti klic, kako se rezultati primerjajo?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Povzetek besedila  
#### Izziv  
Povzemite besedilo tako, da na konec besedilnega odstavka dodate 'tl;dr:'. Opazite, kako model razume, kako izvesti več nalog brez dodatnih navodil. Lahko eksperimentirate z bolj opisnimi pozivi kot tl;dr, da spremenite vedenje modela in prilagodite povzemanje, ki ga prejmete(3).  

Zadnje delo je pokazalo pomembne izboljšave pri mnogih nalogah in merilih NLP z vnaprejšnjim učenjem na velikem besedilnem korpusu, ki mu sledi fino prilagajanje na specifični nalogi. Čeprav je arhitektura ponavadi nalogi agnostična, ta metoda še vedno zahteva nalogam specifične podatkovne nize za fino prilagajanje v tisočih ali desettisočih primerih. V nasprotju s tem ljudje običajno lahko opravijo novo jezikovno nalogo le na podlagi nekaj primerov ali preprostih navodil - nekaj, s čimer se trenutni NLP sistemi še vedno večinoma težko soočajo. Tukaj pokažemo, da znatno povečanje velikosti jezikovnih modelov močno izboljša nalogi agnastične, maloštudijske zmogljivosti, včasih celo doseže konkurenčnost s predhodnimi najboljšimi metodami za fino prilagajanje.  



Tl;dr


# Vaje za več primerov uporabe  
1. Povzetek besedila  
2. Razvrščanje besedila  
3. Ustvari nova imena izdelkov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Razvrsti besedilo  
#### Izziv  
Razvrsti elemente v kategorije, podane ob izvajanju. V naslednjem primeru podamo tako kategorije kot besedilo za razvrščanje v pozivu (*playground_reference). 

Povpraševanje stranke: Pozdravljeni, ena od tipk na moji prenosni tipkovnici se je pred kratkim zlomila in potrebujem zamenjavo:

Razvrščena kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Ustvari nove imena izdelkov
#### Izziv
Ustvarite imena izdelkov iz vzorčnih besed. V pozivu vključimo informacije o izdelku, za katerega bomo ustvarjali imena. Prav tako navedemo podoben primer, da pokažemo vzorec, ki ga želimo dobiti. Nastavili smo tudi visoko vrednost temperature, da povečamo naključnost in bolj inovativne odgovore.

Opis izdelka: Domači aparat za pripravo mlečnih napitkov
Semenske besede: hiter, zdrav, kompakten.
Imena izdelkov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis izdelka: Par čevljev, ki ustreza katerikoli velikosti stopala.
Semenske besede: prilagodljiv, prilega, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najboljše prakse za fino nastavljanje modelov GPT za razvrščanje besedila](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za več pomoči  
[Ekipa za komercializacijo OpenAI](AzureOpenAITeam@microsoft.com) 


# Prispevkarji
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
